# 04 · AutoML 모델 비교

초기 100사이클 피처로 Cycle Life 회귀. 개발 37셀/18충전정책에서만 선택하며, Batch 1 Hold-out 9셀과 Batch 2·3은 선택 후 평가한다. 기존 EDA/실험에서 이미 보았으므로 완전 미열람 테스트는 아니다. **PDF 보고서는 생성하지 않는다.**

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
A = ROOT / "results/automl"
summary = json.loads((A / "run_summary.json").read_text())
summary["selection"]

## 탐색 조건

Ridge, ElasticNet, SVR, LightGBM, XGBoost, CatBoost, Random Forest, Extra Trees의 8개 계열. S1(1개), S2(4개), S3(7개) 피처를 각각 비교했다. 각 탐색은 최대 20초/160 trial, 모든 계열을 순환하는 round-robin 방식이다. log 수명을 학습하지만 **원래 사이클 단위 MAPE**로 선택한다. 그룹 4-fold 내부 CV, 5-fold 외부 CV를 사용한다. 스케일링/결측 대치는 선형·SVR의 각 fold 안에서만 fit하고, 트리 입력은 완전한 수치 피처만 사용한다.

In [ ]:
leaderboard = pd.read_csv(A / "final_leaderboard.csv")
display(leaderboard[["model", "feature_set", "internal_CV_MAPE", "params"]])
display(Image(filename=str(A / "leaderboard.png")))

**코멘트:** 위 순위는 탐색에 사용한 내부 CV 점수라 낙관적일 수 있다. 아래 외부 CV는 피처·모델·파라미터 선택 전체를 매 fold 안에서 반복한 결과다. 각 모델 계열을 독립적으로 nested CV 평가한 순위는 아니며 AutoML 선택 절차의 평가다. 시간 예산 기반 검색은 하드웨어/실행 환경에 따라 최종 파라미터가 달라질 수 있다.

In [ ]:
display(pd.read_csv(A / "nested_cv_folds.csv"))
display({k: summary[k] for k in ["nested_CV_MAPE_mean", "nested_CV_MAPE_SD", "pooled_OOF_MAPE", "seconds"]})

## 모델 계열별 외부 CV 순위

각 계열의 피처/튜닝 선택을 내부 fold에서 수행한 뒤 별도 외부 fold에서 평가했다. 이 순위를 보고 다시 최종 모델을 선택하면 추가 선택 편향이 생기므로, 이미 정한 내부 CV 선정 모델은 유지한다. 작은 표본에서의 점추정 순위이며 통계적 우열을 확정하지 않는다.

In [ ]:
display(pd.read_csv(A / "family_nested_cv_ranking.csv"))

In [ ]:
display(pd.read_csv(A / "model_comparison.csv"))
display(Image(filename=str(A / "family_comparison.png")))

**코멘트:** CatBoost는 계열별 외부 CV 8.63%로 가장 낮지만 Batch 2 MAPE는 43.44%다. Random Forest는 Batch 2 29.01%, 선정 SVR은 30.32%다. Batch 1 CV 순위와 다른 배치의 순위가 달라 배치 간 분포 이동을 의심할 수 있다. 이 비교로 테스트에 맞춰 모델을 재선정하지 않는다. 새 계열을 도입하는 것만으로 Batch 2 성능 문제가 해결되지 않았다.

## 선택을 고정한 뒤 Hold-out / Batch 2·3 평가

In [ ]:
display(pd.read_csv(A / "model_performance.csv"))
display(pd.read_csv(A / "baseline_comparison.csv"))
display(Image(filename=str(A / "baseline_comparison.png")))

**코멘트:** train 행은 재대입 성능이며 Train(CV) 대신 쓰지 않는다. 기존 대비 변화는 관측된 차이이지 유의한 향상을 보장하지 않는다. Batch 2·3 점수로 재선정/교정하지 않았다. 9.1% 원논문 목표는 데이터 매핑/종료 기준이 동일한지 확인되지 않아 참고치다.

In [ ]:
import joblib
from src.preprocess import load_data
df = load_data(ROOT / "data/battery_modeling_ready.parquet")
model = joblib.load(A / "final_model.joblib")
pred = model.predict(df)
assert np.isfinite(pred).all() and (pred > 0).all()
display(pd.DataFrame({"battery_id": df.battery_id, "predicted_cycle_life": pred}).head())

## Colab에서 다시 실행

ZIP을 `/content/ess-battery-project`로 풀고 프로젝트 루트를 작업 경로로 지정한다. 아래 셀은 선택 실행이며 저장된 결과를 보는 것에는 불필요하다. joblib은 신뢰할 수 있는 본 프로젝트 파일만 로드한다.

In [ ]:
# %cd /content/ess-battery-project
# %pip install -r requirements-automl.txt
# !python -m src.automl --seconds 20
# macOS에서 LightGBM 라이브러리 오류가 나면 brew install libomp